# RabTech Task 4 — Supervised Classification / Regression Modeling & Tuning

## UCI Adult (Census Income) — Classification

This notebook continues the **same UCI Adult dataset and feature-engineering approach used in Task 3**. It is designed to satisfy the Task 4 brief: four distinct model architectures, GridSearchCV, stratified K-fold validation, Precision/Recall/F1/ROC-AUC, confusion matrices, ROC-AUC comparison, validation-based champion selection, feature importance, and serialized `.joblib` model.

> **Reproducibility:** The first cell installs the three runtime packages that commonly cause import errors (`xgboost`, `lightgbm`, `ucimlrepo`). If your environment already has them, pip simply reports them as satisfied.


In [ ]:
# Run this cell first. It fixes the common ModuleNotFoundError for XGBoost/LightGBM/UCI loader.
%pip install -q xgboost lightgbm ucimlrepo


In [ ]:
import warnings
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve,
    classification_report
)
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
ARTIFACT_DIR = Path('artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)
print('Imports successful: XGBoost + LightGBM + UCI loader are available.')


## 1. Dataset and provenance

The UCI Adult dataset contains 48,842 instances and 14 input features and is used to predict whether annual income exceeds $50K/year. Task 3 used the UCI Machine Learning Repository source.

**Official source:** https://archive.ics.uci.edu/dataset/2/adult

**Citation:** Becker, B. & Kohavi, R. (1996). Adult [Dataset]. UCI Machine Learning Repository. DOI: https://doi.org/10.24432/C5XW20

The notebook loads the dataset through `ucimlrepo`, exactly matching the Task 3 dataset family.


In [ ]:
# 2. Load the same UCI Adult dataset used in Task 3
adult = fetch_ucirepo(id=2)
X_raw = adult.data.features.copy()
y_raw = adult.data.targets.copy()

# Normalize the target to a clean binary Series.
target_col = y_raw.columns[0]
y = y_raw[target_col].astype(str).str.strip().str.replace('.', '', regex=False)
y = y.map({'<=50K': 0, '>50K': 1})
if y.isna().any():
    raise ValueError('Unexpected target labels were found. Inspect y_raw.')

print('Raw feature shape:', X_raw.shape)
print('Target distribution:')
display(y.value_counts().rename(index={0:'<=50K',1:'>50K'}).to_frame('count'))
display(X_raw.head())


In [ ]:
# 3. Train/test split BEFORE any learned preprocessing.
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print('Training rows:', len(X_train))
print('Testing rows :', len(X_test))


## 4. Task-3-compatible reusable feature engineering + preprocessing

The same four engineered features from Task 3 are retained:
- `capital_net`
- `hours_age_ratio`
- `capital_gain_flag`
- `age_band`

`education` is removed because `education-num` already supplies the ordered numeric representation. All transformations are inside the pipeline, so GridSearchCV never fits preprocessing on a validation fold before the fold split.


In [ ]:
ADULT_COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country'
]

NUMERIC_RAW = [
    'age', 'fnlwgt', 'education-num',
    'capital-gain', 'capital-loss', 'hours-per-week'
]

class AdultFeatureEngineer(BaseEstimator, TransformerMixin):
    """Clean raw Adult columns and create leakage-safe derived features."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X.columns = ADULT_COLUMNS
        for col in X.columns:
            if X[col].dtype == 'object':
                X[col] = X[col].astype(str).str.strip().replace({'?': np.nan})
        for col in NUMERIC_RAW:
            X[col] = pd.to_numeric(X[col], errors='coerce')

        X = X.drop(columns=['education'])
        X['capital_net'] = X['capital-gain'] - X['capital-loss']
        X['hours_age_ratio'] = X['hours-per-week'] / (X['age'] + 1.0)
        X['capital_gain_flag'] = (X['capital-gain'] > 0).astype(int)
        X['age_band'] = pd.cut(
            X['age'], bins=[0,25,35,50,np.inf],
            labels=['<=25','26-35','36-50','51+'], include_lowest=True
        )
        return X

    def get_feature_names_out(self, input_features=None):
        return np.array([
            'age','workclass','fnlwgt','education-num','marital-status',
            'occupation','relationship','race','sex','capital-gain','capital-loss',
            'hours-per-week','native-country','capital_net','hours_age_ratio',
            'capital_gain_flag','age_band'
        ], dtype=object)

engineered_numeric = [
    'age','fnlwgt','education-num','capital-gain','capital-loss',
    'hours-per-week','capital_net','hours_age_ratio','capital_gain_flag'
]
engineered_categorical = [
    'workclass','marital-status','occupation','relationship',
    'race','sex','native-country','age_band'
]

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler(with_mean=False))
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, engineered_numeric),
    ('categorical', categorical_pipeline, engineered_categorical)
], remainder='drop', verbose_feature_names_out=False)

print('Feature engineering and preprocessing objects created successfully.')


## 5. Majority-class baseline

This is a simple no-skill reference. It is reported before tuning and is **not** presented as a trained model.


In [ ]:
majority_class = int(y_train.mode().iloc[0])
baseline_pred = np.full(len(y_test), majority_class)
baseline = {
    'accuracy': accuracy_score(y_test, baseline_pred),
    'precision': precision_score(y_test, baseline_pred, zero_division=0),
    'recall': recall_score(y_test, baseline_pred, zero_division=0),
    'f1': f1_score(y_test, baseline_pred, zero_division=0)
}
print('Majority class:', majority_class)
print(pd.Series(baseline).round(4))


## 6. Four distinct model architectures + stratified 5-fold GridSearchCV

The four architectures are: Logistic Regression, Random Forest, XGBoost and LightGBM. The primary tuning metric is F1-score.


In [ ]:
models = {
    'Logistic Regression': (
        LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
        {'model__C':[0.1,1.0,10.0], 'model__solver':['liblinear']}
    ),
    'Random Forest': (
        RandomForestClassifier(n_estimators=150, n_jobs=-1, class_weight='balanced', random_state=RANDOM_STATE),
        {'model__n_estimators':[100,150], 'model__max_depth':[None,20]}
    ),
    'XGBoost': (
        XGBClassifier(objective='binary:logistic', eval_metric='logloss', tree_method='hist', random_state=RANDOM_STATE, n_jobs=2),
        {'model__n_estimators':[100,200], 'model__max_depth':[3,5], 'model__learning_rate':[0.05]}
    ),
    'LightGBM': (
        LGBMClassifier(objective='binary', verbosity=-1, random_state=RANDOM_STATE, n_jobs=2),
        {'model__n_estimators':[100,200], 'model__num_leaves':[15,31], 'model__learning_rate':[0.05]}
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
searches = {}
validation_rows = []

for name, (estimator, param_grid) in models.items():
    pipe = Pipeline([
        ('feature_engineering', AdultFeatureEngineer()),
        ('preprocessor', preprocessor),
        ('model', estimator)
    ])
    search = GridSearchCV(
        pipe, param_grid=param_grid, scoring='f1', cv=cv,
        n_jobs=-1, refit=True, return_train_score=False
    )
    search.fit(X_train, y_train)
    searches[name] = search
    validation_rows.append({
        'model': name,
        'cv_f1_mean': search.best_score_,
        'best_params': json.dumps(search.best_params_, default=str)
    })

validation_results = pd.DataFrame(validation_rows).sort_values(
    ['cv_f1_mean','model'], ascending=[False,True]
).reset_index(drop=True)
display(validation_results)
validation_results.to_csv(ARTIFACT_DIR/'validation_model_comparison.csv', index=False)


## 7. Held-out test evaluation

The test set is used only after hyperparameter tuning. This prevents the test set from influencing model selection.


In [ ]:
test_rows = []
predictions = {}
for name, search in searches.items():
    model = search.best_estimator_
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:,1]
    predictions[name] = (pred, prob)
    test_rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test,pred),
        'precision': precision_score(y_test,pred,zero_division=0),
        'recall': recall_score(y_test,pred,zero_division=0),
        'f1': f1_score(y_test,pred,zero_division=0),
        'roc_auc': roc_auc_score(y_test,prob)
    })

test_results = pd.DataFrame(test_rows).sort_values(
    ['f1','roc_auc'], ascending=False
).reset_index(drop=True)
display(test_results.round(4))
test_results.to_csv(ARTIFACT_DIR/'model_comparison.csv', index=False)


In [ ]:
# 8. Confusion matrices for all four models
for name, (pred, prob) in predictions.items():
    fig, ax = plt.subplots(figsize=(5,4))
    ConfusionMatrixDisplay(
        confusion_matrix(y_test,pred), display_labels=['<=50K','>50K']
    ).plot(ax=ax, values_format='d', colorbar=False)
    ax.set_title(f'Confusion Matrix — {name}')
    fig.tight_layout()
    filename = name.lower().replace(' ','_') + '_confusion_matrix.png'
    fig.savefig(ARTIFACT_DIR/filename, dpi=180, bbox_inches='tight')
    plt.show()


In [ ]:
# 9. ROC-AUC comparison curve
plt.figure(figsize=(8,6))
for name, (pred, prob) in predictions.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'--',label='Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-AUC Comparison — Held-out Test Set')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR/'roc_auc_comparison.png', dpi=180, bbox_inches='tight')
plt.show()


## 10. Champion model selection

The champion is selected from the **validation results** using the highest mean 5-fold F1-score. The held-out test metrics are reported afterward for final evaluation.


In [ ]:
champion_name = validation_results.iloc[0]['model']
champion_search = searches[champion_name]
champion_model = champion_search.best_estimator_
champion_test = test_results[test_results['model']==champion_name].iloc[0]

print('CHAMPION MODEL:', champion_name)
print('Validation CV F1:', round(champion_search.best_score_,4))
print('Test Accuracy   :', round(champion_test['accuracy'],4))
print('Test Precision  :', round(champion_test['precision'],4))
print('Test Recall     :', round(champion_test['recall'],4))
print('Test F1         :', round(champion_test['f1'],4))
print('Test ROC-AUC    :', round(champion_test['roc_auc'],4))
print('Best parameters :', champion_search.best_params_)


## 11. Feature importance

For tree-based champions, native feature importance is used. For Logistic Regression, absolute coefficient magnitude is used.


In [ ]:
feature_names = champion_model.named_steps['preprocessor'].get_feature_names_out()
estimator = champion_model.named_steps['model']
if hasattr(estimator,'feature_importances_'):
    importance = estimator.feature_importances_
    importance_method = 'tree feature_importances_'
elif hasattr(estimator,'coef_'):
    importance = np.abs(estimator.coef_[0])
    importance_method = 'absolute logistic-regression coefficients'
else:
    raise ValueError('Champion model does not expose feature importance.')

importance_df = pd.DataFrame({'feature':feature_names,'importance':importance}).sort_values(
    'importance', ascending=False
).reset_index(drop=True)
print('Method:', importance_method)
display(importance_df.head(20))
importance_df.to_csv(ARTIFACT_DIR/'champion_feature_importance.csv', index=False)

top = importance_df.head(15).sort_values('importance')
plt.figure(figsize=(8,6))
plt.barh(top['feature'], top['importance'])
plt.xlabel('Importance')
plt.title(f'Top Feature Importance — {champion_name}')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR/'champion_feature_importance.png', dpi=180, bbox_inches='tight')
plt.show()


## 12. Serialize the complete champion pipeline

The serialized object contains feature engineering + preprocessing + the fitted champion model. It can therefore receive raw Adult feature rows at inference time.


In [ ]:
model_path = ARTIFACT_DIR/'champion_model.joblib'
joblib.dump(champion_model, model_path)
metadata = {
    'champion_model': champion_name,
    'selection_metric': 'mean 5-fold stratified CV F1-score',
    'random_state': RANDOM_STATE,
    'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'target': 'income',
    'dataset': 'UCI Adult / Census Income',
    'model_path': str(model_path)
}
with open(ARTIFACT_DIR/'run_metadata.json','w',encoding='utf-8') as f:
    json.dump(metadata,f,indent=2)
print('Serialized champion:', model_path)


## 13. Final Task 4 checklist

| Requirement from brief | Evidence in notebook |
|---|---|
| 4 distinct model architectures | Logistic Regression, Random Forest, XGBoost, LightGBM |
| Hyperparameter optimization | GridSearchCV |
| Stratified K-fold | StratifiedKFold, 5 folds |
| Precision | Model comparison table |
| Recall | Model comparison table |
| F1-score | Model comparison + validation selection |
| ROC-AUC | Model comparison + ROC curve |
| Confusion Matrix | Four generated matrices |
| Champion model | Validation-based selection |
| Feature importance | Top feature table + plot |
| Serialized model | `artifacts/champion_model.joblib` |
| Leakage control | Split before fit + transformations inside pipeline |
